# Gaffer · Day 5 — Fine-tuning a football detector (Colab T4)

Goal: replace the base `yolo11n.pt` (COCO: person + sports-ball only) with a
football-aware `yolov11_football.pt` that knows **player / goalkeeper / referee / ball**.

> **Runtime → Change runtime type → T4 GPU** before running anything.

---
## Dataset: Roboflow `football-players-detection`

| | |
|---|---|
| **Source** | Roboflow Universe (`roboflow-jvuqo/football-players-detection`) |
| **Classes** | `ball`, `goalkeeper`, `player`, `referee` (4) |
| **Images** | ~600 annotated broadcast frames (train/valid/test split) |
| **Labels** | YOLO format bounding boxes |
| **License** | CC BY 4.0 (free) |

**Why this one:** its four classes are *exactly* Gaffer's class scheme — no other
free dataset maps this cleanly onto the existing pipeline. The base COCO model
cannot distinguish goalkeepers/referees and barely detects the ball; this fixes
all three in one fine-tune.

**Limitations to keep in mind:**
- **Ball is heavily under-represented** (one tiny box per frame vs ~20 players) → expect lower ball mAP and watch for ball false-negatives.
- **Small dataset (~600 imgs)** → real overfitting risk; we use early-stopping (`patience`) and `close_mosaic`.
- **Mostly one broadcast style** → domain gap vs your tactical-cam clips. The local A/B test is what proves it actually generalises to *your* footage.
- Bigger alternative for later: **SoccerNet** (orders of magnitude larger, but needs more preprocessing).


## 0 · GPU check


In [ ]:
!nvidia-smi

## 1 · Install


In [ ]:
!pip install -q ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 2 · Download the dataset

Get a **free** Roboflow API key: https://app.roboflow.com → Settings → API key.
Paste it below. The export format must be **yolov11**.


In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''  # <-- paste your free key here
VERSION = 12           # dataset version on Roboflow Universe

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
dataset = project.version(VERSION).download('yolov11')
DATASET_DIR = dataset.location
print('downloaded to', DATASET_DIR)

## 3 · ⚠️ Remap classes to Gaffer's order — DO NOT SKIP

Roboflow ships **alphabetical** order: `ball=0, goalkeeper=1, player=2, referee=3`.
Gaffer's pipeline (`gaffer/config.py`) expects `player=0, goalkeeper=1, referee=2, ball=3`
and `detector.py` trusts the model's raw class index directly.

If we train without remapping, the model emits `0` for a ball and **Gaffer draws it
as a player** — a silent, catastrophic mislabel. The cell below rewrites every label
file and `data.yaml` to Gaffer's order. (Mirrors `scripts/train_detector.py`.)


In [ ]:
from pathlib import Path
import yaml

GAFFER_NAMES = ['player', 'goalkeeper', 'referee', 'ball']  # index == gaffer id
ROBOFLOW_TO_GAFFER = {0: 3, 1: 1, 2: 0, 3: 2}  # ball,gk,player,ref -> gaffer

def remap_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir)
    yaml_path = dataset_dir / 'data.yaml'
    cfg = yaml.safe_load(yaml_path.read_text())
    if cfg.get('gaffer_remapped'):
        print('already remapped'); return
    files = boxes = 0
    for txt in dataset_dir.rglob('labels/*.txt'):
        out = []
        for line in txt.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            p = line.split()
            c = int(p[0])
            if c not in ROBOFLOW_TO_GAFFER:
                continue
            p[0] = str(ROBOFLOW_TO_GAFFER[c])
            out.append(' '.join(p)); boxes += 1
        txt.write_text('\n'.join(out) + ('\n' if out else ''))
        files += 1
    cfg['names'] = GAFFER_NAMES
    cfg['nc'] = len(GAFFER_NAMES)
    cfg['gaffer_remapped'] = True
    yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f'remapped {files} files, {boxes} boxes -> {GAFFER_NAMES}')

remap_dataset(DATASET_DIR)
DATA_YAML = str(Path(DATASET_DIR) / 'data.yaml')

### Sanity-check the remap

Confirm `data.yaml` names are in Gaffer order and a sample label uses the new ids.


In [ ]:
print(open(DATA_YAML).read())
sample = next(Path(DATASET_DIR).rglob('labels/*.txt'))
print('sample label', sample.name)
print(sample.read_text()[:300])

## 4 · Train YOLO11n (first pass)

`imgsz=1280` matters: players and especially the ball are small. Mosaic is
disabled for the final 10 epochs (`close_mosaic`) for cleaner boxes.


In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolo11n.pt')
res_n = model_n.train(
    data=DATA_YAML, epochs=50, imgsz=1280, batch=16, device=0,
    project='runs/gaffer_detector', name='yolo11n_football',
    patience=15, fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, plots=True)

### Per-class metrics + confusion matrix (YOLO11n)


In [ ]:
from IPython.display import Image, display
m = model_n.val(data=DATA_YAML, device=0, plots=True)
b = m.box
print(f"{'class':<12}{'mAP50':>8}{'mAP50-95':>10}{'P':>8}{'R':>8}")
print(f"{'all':<12}{b.map50:>8.3f}{b.map:>10.3f}{b.mp:>8.3f}{b.mr:>8.3f}")
for row, cid in enumerate(b.ap_class_index):
    print(f"{model_n.names[int(cid)]:<12}{b.ap50[row]:>8.3f}{b.ap[row]:>10.3f}{b.p[row]:>8.3f}{b.r[row]:>8.3f}")
display(Image(filename=str(Path(res_n.save_dir) / 'confusion_matrix.png')))

## 5 · Decision gate — is YOLO11s worth it?

Train the slightly larger `yolo11s` **only if** YOLO11n's headline mAP50 is
underwhelming (say < 0.85 on `player`) or ball recall is poor. On a T4, `s` is
~2× slower at inference — which matters because Gaffer runs on an Intel Arc iGPU,
not a GPU. Prefer `n` unless `s` is clearly better. Run the next cell to compare;
skip it if `n` is already strong.


In [ ]:
model_s = YOLO('yolo11s.pt')
res_s = model_s.train(
    data=DATA_YAML, epochs=50, imgsz=1280, batch=16, device=0,
    project='runs/gaffer_detector', name='yolo11s_football',
    patience=15, fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, plots=True)
ms = model_s.val(data=DATA_YAML, device=0)
print(f'YOLO11n mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}')
print(f'YOLO11s mAP50={ms.box.map50:.3f} mAP50-95={ms.box.map:.3f}')
print('Pick s only if the gain justifies ~2x slower CPU/iGPU inference.')

## 6 · Export the winner → `weights/yolov11_football.pt`

Pick whichever run won, then download `best.pt`. Place it in the repo at
**`weights/yolov11_football.pt`** — that's the exact path `gaffer/config.py`
(`DETECTION_MODEL_PATH`) and the auto-detect in `FootballDetector` look for.


In [ ]:
from google.colab import files
WINNER = res_n.save_dir   # change to res_s.save_dir if YOLO11s won
best = Path(WINNER) / 'weights' / 'best.pt'
print('downloading', best)
files.download(str(best))
# also grab the metrics + confusion matrix for the repo record
files.download(str(Path(WINNER) / 'results.csv'))
files.download(str(Path(WINNER) / 'confusion_matrix.png'))

## 7 · Back on the local machine — prove it helps

mAP on the validation set proves the model learned the dataset. It does **not**
prove it helps Gaffer's pipeline on *your* tactical clips. Run the local A/B gate:

```bash
# place the downloaded best.pt:
#   weights/yolov11_football.pt
uv run python scripts/ab_compare_detectors.py \
    data/test_clips/tactical_playlist_1.mp4 --start 30 --duration 60
```

Success criterion: the fine-tuned model wins ≥3/4 headline checks — fewer unique
track IDs, more tracks ≥10s, higher persistence, and it actually detects the ball.

Then regenerate the v0.1 demo to *see* the difference:
```bash
uv run python gaffer/pipeline.py data/test_clips/tactical_playlist_1.mp4 \
    -o outputs/v0_2_finetuned_demo.mp4
```
